**Personal Expense Monitoring System**

Week 4

In [0]:
EXPENSES_PATH = "/Volumes/sasi_008/default/training_files/expenses_clean.csv"   
USERS_PATH    = "/Volumes/sasi_008/default/training_files/users.csv"
OUT_DIR       = "/Volumes/sasi_008/default/training_files/output"   

In [0]:
SAVINGS_RATE_MIN = 0.10   
SPIKE_RATIO      = 1.5 

In [0]:
from pyspark.sql import functions as F
 
expenses = (spark.read.csv(EXPENSES_PATH, header=True, inferSchema=True)
            .withColumn("date", F.to_date("date")))
users = spark.read.csv(USERS_PATH, header=True, inferSchema=True)
 
print("expenses:", expenses.count(), "| users:", users.count())
display(expenses.limit(5))
 

expenses: 566 | users: 5


expense_id,user_id,category,amount,date,description,payment_method,month
1,1,Rent,1060.5,2026-04-01,rent purchase,UPI,2026-04-01
2,1,Utilities,79.66,2026-04-01,utilities purchase,Debit Card,2026-04-01
7,1,Shopping,103.82,2026-04-02,shopping purchase,Debit Card,2026-04-01
8,1,Groceries,52.01,2026-04-03,groceries purchase,Credit Card,2026-04-01
4,1,Dining,40.55,2026-04-05,dining purchase,Cash,2026-04-01


In [0]:
combined = (expenses.join(users, "user_id", "inner")
            .withColumn("month", F.date_format("date", "yyyy-MM")))
 
monthly = (combined.groupBy("user_id", "name", "month", "monthly_income", "monthly_budget")
           .agg(F.round(F.sum("amount"), 2).alias("total_spend"),
                F.count("*").alias("txn_count"),
                F.round(F.max("amount"), 2).alias("largest_txn")))

In [0]:
from pyspark.sql import Window
 
med = monthly.groupBy("user_id").agg(F.percentile_approx("total_spend", 0.5).alias("median_spend"))
 
summary = (monthly.join(med, "user_id")
           .withColumn("savings", F.round(F.col("monthly_income") - F.col("total_spend"), 2))
           .withColumn("savings_rate", F.round(F.col("savings") / F.col("monthly_income"), 3))
           .withColumn("over_budget", F.col("total_spend") > F.col("monthly_budget"))
           .withColumn("spike_ratio", F.round(F.col("total_spend") / F.col("median_spend"), 2))
           .withColumn("alert",
               F.when(F.col("spike_ratio") > SPIKE_RATIO, "UNUSUAL_SPIKE")
                .when(F.col("over_budget"), "OVER_BUDGET")
                .when(F.col("savings_rate") < SAVINGS_RATE_MIN, "LOW_SAVINGS")
                .otherwise("OK"))
           .drop("median_spend")
           .orderBy("user_id", "month"))
 
display(summary)

user_id,name,month,monthly_income,monthly_budget,total_spend,txn_count,largest_txn,savings,savings_rate,over_budget,spike_ratio,alert
1,User 1,2026-04,3535,2614.0,4764.79,19,2828.0,-1229.79,-0.348,true,2.39,UNUSUAL_SPIKE
1,User 1,2026-05,3535,2614.0,2134.99,19,1060.5,1400.01,0.396,false,1.07,OK
1,User 1,2026-06,3535,2614.0,1952.08,19,1060.5,1582.92,0.448,false,0.98,OK
1,User 1,2026-07,3535,2614.0,2099.19,18,1060.5,1435.81,0.406,false,1.05,OK
1,User 1,2026-08,3535,2614.0,1730.04,19,1060.5,1804.96,0.511,false,0.87,OK
1,User 1,2026-09,3535,2614.0,1991.01,19,1060.5,1543.99,0.437,false,1.0,OK
2,User 2,2026-04,7643,4730.0,3534.11,19,2292.9,4108.89,0.538,false,1.09,OK
2,User 2,2026-05,7643,4730.0,3324.37,19,2292.9,4318.63,0.565,false,1.02,OK
2,User 2,2026-06,7643,4730.0,3235.2,19,2292.9,4407.8,0.577,false,1.0,OK
2,User 2,2026-07,7643,4730.0,3256.09,19,2292.9,4386.91,0.574,false,1.0,OK


In [0]:
(summary.write.format("delta").mode("overwrite").save(f"{OUT_DIR}/monthly_summary_delta"))
(summary.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{OUT_DIR}/monthly_summary_csv"))

In [0]:
summary.write.format("delta").mode("overwrite").saveAsTable("sasi_008.default.expense_monthly_summary")

In [0]:
check = spark.read.format("delta").load(f"{OUT_DIR}/monthly_summary_delta")
print("rows in Delta:", check.count())
display(check.groupBy("alert").count())

rows in Delta: 30


alert,count
UNUSUAL_SPIKE,2
OK,28


In [0]:
%sql
SELECT
  month,
  ROUND(SUM(total_spend), 2) AS total_spend,
  SUM(CASE WHEN alert <> 'OK' THEN 1 ELSE 0 END) AS alerts
FROM sasi_008.default.expense_monthly_summary
GROUP BY month
ORDER BY month;

month,total_spend,alerts
2026-04,22170.96,2
2026-05,13723.42,0
2026-06,14073.78,0
2026-07,13637.68,0
2026-08,13256.74,0
2026-09,13266.44,0
